In [ ]:
DRY_RUN = True #für Update auf True setzen

if DRY_RUN:
    raise SystemExit

# Fine-Tuning, Quantisierung und wissenschaftliche Evaluierung

**LoRA/PEFT · GGUF + Ollama · Drei-Varianten-Vergleich mit Bootstrap-CIs und Wilcoxon-Tests**

Dieses Notebook ist ein reiner Orchestrator. Die gesamte Logik liegt im Package
`src/finetune/`; jede Codezelle unten ist ein duenner Aufruf in dieses Package.

---

## Was du anfassen musst

**Nur Zelle 0.3.** Dort stehen Basismodell, Quantisierungsstufe, Hyperparameter und
Evaluierungsumfang. Alles andere — Pfade, Artefaktnamen, Ollama-Modellnamen,
LoRA-Zielmodule, Chat-Templates — wird daraus abgeleitet. Im uebrigen Notebook
steht kein einziger Modellname und kein absoluter Pfad als Literal.

## Ablauf

| Phase | Was passiert | Ergebnis auf der Platte |
|-------|--------------|-------------------------|
| **0** | Package spiegeln, Konfiguration setzen, Abhaengigkeiten | — |
| **1** | Umgebung pruefen, HF-Login, Verzeichnisse | Verzeichnisbaum |
| **2** | Splits spiegeln, Statistik, Kontaminationspruefung | `data/*.json` |
| **3** | Preflight: Ressourcenbudget + 6 Integritaetschecks | — |
| **4** | LoRA-Fine-Tuning | `*-peft/` |
| **5** | Drei Modellvarianten erzeugen und in Ollama registrieren | `*.gguf` |
| **6** | Antworten generieren, Metriken, Judge, Statistik | `results/*.json`, PNGs |
| **7** | Reproduzierbarkeit: Manifest, Requirements | `run_config.json` |
| **A/B** | Diagnose und Artefakt-Sicherung — standardmaessig aus | — |

## Die drei Modellvarianten

Der Kern der Evaluierung: Quantisierungsfehler und Fine-Tuning-Effekt werden
**getrennt** gemessen, indem drei Varianten mit identischen Fragen, identischem
System-Prompt und identischen Sampling-Parametern befragt werden.

| Variante | Was sie ist | Was der Vergleich zur Vorgaengerin zeigt |
|----------|-------------|------------------------------------------|
| `original` | Basismodell, unquantisiert (fp16, Transformers) | Referenz |
| `base_quant` | dasselbe Modell als GGUF, ohne Fine-Tuning | **reiner Quantisierungsfehler** |
| `finetuned_quant` | GGUF nach LoRA-Merge | **Fine-Tuning-Effekt** unter Quantisierung |

## Kaggle-Grenzen

`/kaggle/input` ist read-only und wird nur gelesen. `/kaggle/working` (~20 GB) haelt
die dauerhaften Artefakte, `/tmp` die grossen Zwischenschritte (fluechtig!).
32 GB RAM, T4 mit 16 GB VRAM. Zelle 0.3 rechnet vor dem ersten teuren Schritt aus,
ob die gewaehlte Kombination hineinpasst, und bricht sonst mit einer konkreten
Empfehlung ab.

## Phase 0 · Bootstrap und Konfiguration

Zelle 0.2 ist die **einzige** Zelle mit eigener Logik. Das ist unvermeidbar:
`config.py` liegt im Package, und das Package muss erst importierbar gemacht werden.
Deshalb steht hier — und nur hier — der Input-Pfad als Literal. Er muss zu
`CFG.paths.input_dir` in Zelle 0.3 passen.

Gespiegelt wird nach `/kaggle/working/src`, weil ein Import direkt aus
`/kaggle/input` die `.pyc`-Caches nicht schreiben kann und Codeaenderungen nicht
wirksam werden.

In [ ]:
# ── Zelle 0.2 · Package spiegeln und importierbar machen ─────────────────
# Einzige Zelle mit eigener Logik (Henne-Ei: config.py liegt im Package).
import importlib
import shutil
import sys
from pathlib import Path

INPUT_SRC = Path("/kaggle/input/datasets/feyssal/finetuning/FineTuning/src")  # read-only
WORK_SRC = Path("/kaggle/working/src")                                        # schreibbar

if INPUT_SRC.exists():
    if WORK_SRC.exists():
        shutil.rmtree(WORK_SRC)
    shutil.copytree(INPUT_SRC, WORK_SRC)
    print(f"✅ Package gespiegelt: {INPUT_SRC}\n                    → {WORK_SRC}")
elif (WORK_SRC / "finetune").is_dir():
    print(f"ℹ️  Kein Input-Package gefunden — nutze die vorhandene Kopie: {WORK_SRC}")
else:
    raise FileNotFoundError(
        f"Weder {INPUT_SRC} noch {WORK_SRC}/finetune gefunden.\n"
        f"   → Dataset an das Notebook anhaengen oder INPUT_SRC anpassen."
    )

# Read-only Pfade aus sys.path entfernen, working copy nach vorn
sys.path = [p for p in sys.path if "/kaggle/input" not in p]
if str(WORK_SRC) not in sys.path:
    sys.path.insert(0, str(WORK_SRC))

# Caches leeren, damit eine frische Kopie auch wirklich neu geladen wird
sys.path_importer_cache.pop(str(WORK_SRC), None)
importlib.invalidate_caches()
for _name in [m for m in sys.modules if m == "finetune" or m.startswith("finetune.")]:
    del sys.modules[_name]

if not (WORK_SRC / "finetune" / "config.py").is_file():
    raise FileNotFoundError(f"config.py fehlt unter {WORK_SRC / 'finetune'}")
print("✅ finetune-Package ist importierbar.")

### Zelle 0.3 · Die Konfigurationszelle

Die einzige Zelle, die du im Normalbetrieb aenderst.

`PipelineConfig` leitet aus `model_key`, `quant_key` und `dataset_tag` alle
Artefaktpfade und Ollama-Modellnamen ab. Ein Modellwechsel ist damit eine
Zeilenaenderung, kein Suchen-und-Ersetzen.

`CFG.validate()` rechnet VRAM-, RAM- und Plattenbedarf gegen die Limits aus
`CFG.runtime` und bricht ab, bevor eine GPU-Stunde verbraucht ist. Die Meldung
nennt die konkrete Gegenmassnahme.

**Verfuegbare Profile:** `gemma-4-e4b`, `gemma-2-2b`, `deepseek-r1-qwen-7b`,
`deepseek-r1-qwen-1_5b`, `llama-3_1-8b`, `qwen2_5-7b` ·
Quantisierung: `Q3_K_M`, `Q4_K_M`, `Q5_K_M`, `Q6_K`, `Q8_0`, `F16`

Ein neues Basismodell traegst du als `ModelProfile` in `src/finetune/config.py`
ein — dort stehen Repo-ID, LoRA-Zielmodule, Mindestversionen und das
Ollama-Template beieinander.

In [ ]:
# ── Zelle 0.3 · KONFIGURATION — hier und nur hier stellst du etwas ein ───
from finetune.config import PipelineConfig, set_config

CFG = PipelineConfig(
    model_key="gemma-4-e4b",      # Basismodell (Profile siehe config.py)
    quant_key="Q4_K_M",           # Quantisierungsstufe fuer den GGUF-Export
    dataset_tag="dfss",           # Kuerzel des Datensatzes, geht in alle Artefaktnamen
)

# ── Pfade (nur anpassen, wenn dein Dataset anders eingehaengt ist) ───────
# CFG.paths.input_dir = Path("/kaggle/input/datasets/<user>/<dataset>/<ordner>")

# ═══ SMOKE-MODUS ════════════════════════════════════════════════════════
# Fuer den ERSTEN Durchlauf auf Kaggle: kuerzt Training (20 Schritte statt
# ~315) und Evaluierung (30 statt 181 Fragen), sodass die GESAMTE Kette in
# 1,5–2 h durchlaeuft statt in 6–9 h. Ein voller Lauf passt nicht in eine
# Kaggle-Session.
#
# ⚠️  Die Zahlen aus einem Smoke-Lauf sind NICHT auswertbar. Vor dem
#     eigentlichen Lauf die Zeile auskommentieren ODER
#     CFG.disable_smoke_mode() aufrufen — das setzt die Werte zurueck UND
#     erzwingt ein neues Training. Zusaetzlich in Zelle 6.2:
#         answers = generate_answers(CFG, force=True)
#
# Was der Smoke-Lauf baut, bleibt nutzbar: llama.cpp, HF-Checkpoint und die
# beiden Basis-Varianten haengen nicht am Training.

CFG.enable_smoke_mode()                # ← fuer den vollen Lauf auskommentieren
# CFG.disable_smoke_mode()             # ← oder das hier aktivieren

# ── Ablaufsteuerung ─────────────────────────────────────────────────────
CFG.training.force_retrain = False     # True → neu trainieren trotz vorhandenem Adapter
CFG.training.resume_training = False   # True → am letzten Checkpoint fortsetzen
CFG.export.force_requantize = False    # True → GGUF neu bauen trotz vorhandener Datei

# ── Training ────────────────────────────────────────────────────────────
CFG.training.epochs = 3
CFG.training.batch_size = 1            # T4: 1 lassen, ueber grad_accum skalieren
CFG.training.grad_accum = 8            # effektive Batchgroesse = batch_size * grad_accum
CFG.training.learning_rate = 2e-4
CFG.training.max_seq_length = 512      # groesster Hebel auf den VRAM-Bedarf

# ── LoRA ────────────────────────────────────────────────────────────────
CFG.lora.r = 16
CFG.lora.alpha = 32
CFG.lora.dropout = 0.05

# ── Evaluierung ─────────────────────────────────────────────────────────
# max_questions und bootstrap_iterations setzt der Smoke-Modus oben bereits;
# hier nur ueberschreiben, wenn du bewusst davon abweichen willst.
CFG.evaluation.variants = ("original", "base_quant", "finetuned_quant")
CFG.evaluation.num_predict = 512       # NICHT unter ~400: sonst werden Referenzantworten
                                       # abgeschnitten und BLEU/ROUGE-L messen Laenge
CFG.evaluation.bertscore_lang = "en"   # Datensatz ist durchgehend englisch
CFG.evaluation.bertscore_model = "roberta-large"

# ── LLM-Judge (0-4 je Dimension) ────────────────────────────────────────
CFG.evaluation.judge_enabled = True
CFG.evaluation.judge_model = "qwen2.5:7b-instruct"   # muss auf Kaggle laufen koennen
                                       # groessere Judges (z. B. "gemma4:31b") sind
                                       # waehlbar, aber auf T4/32 GB nicht praktikabel

# ── Statistik ───────────────────────────────────────────────────────────
CFG.evaluation.confidence = 0.95
CFG.evaluation.seed = 42

# ── Anhaenge (standardmaessig aus) ──────────────────────────────────────
CFG.debug = False                      # Anhang A: Diagnoseausgaben
CFG.publish_artifacts = False          # Anhang B: Artefakte als Kaggle-Dataset sichern

# ── Aktivieren, pruefen, anzeigen ───────────────────────────────────────
set_config(CFG)          # setzt zusaetzlich CUDA_VISIBLE_DEVICES und OLLAMA_MODELS
CFG.validate()           # bricht ab, wenn die Kombination nicht in die Limits passt
print(CFG.summary())

_warning = CFG.smoke_warning()
if _warning:
    print("\n" + _warning)

### Zelle 0.4 · Abhaengigkeiten

`install_dependencies()` installiert nur, was fehlt oder in der falschen Version
dasteht — zweimal ausfuehren kostet nichts. Die Paketliste kommt aus
`CFG.pip_packages`.

**torch wird bewusst nicht angefasst.** Die auf Kaggle vorinstallierte CUDA-Build
passt zum Treiber der Instanz; ein `pip install torch` zieht eine inkompatible
Version und kostet mehrere GB.

Falls danach ein Import scheitert (typisch nach einem `transformers`-Upgrade in
einer bereits laufenden Session): Kernel neu starten und ab Zelle 0.2 weiter.

In [ ]:
# ── Zelle 0.4 · Abhaengigkeiten installieren (idempotent) ────────────────
from finetune.bootstrap import install_dependencies, verify_package_origin

install_dependencies(CFG)
verify_package_origin(CFG)

## Phase 1 · Umgebung, Authentifizierung, Verzeichnisse

Zelle 1.2 ist die **einzige Importzelle** des Notebooks. Alle Package-Funktionen
werden hier nach Modulen gruppiert geholt, sodass auf einen Blick sichtbar ist,
welches Modul was beisteuert — und welche Komponente man austauschen wuerde.

Danach vier Schritte, die zusammen unter einer Minute brauchen und die haeufigsten
Abbruchgruende ausschliessen, bevor GPU-Zeit verbraucht wird:

- **`report_environment()`** — Paketversionen, GPU, RAM, Plattenplatz. Wirft
  bewusst **nicht**: eine Diagnosefunktion soll berichten, nicht abbrechen. Der
  harte Versionscheck sitzt im Preflight (Phase 3).
- **`authenticate()`** — Token aus den Kaggle-Secrets, exportiert `HF_TOKEN` und
  `HUGGING_FACE_HUB_TOKEN`. Ob ein fehlender Token ein Abbruch ist, entscheidet
  `ModelProfile.gated`: Gemma und LLaMA ja, DeepSeek und Qwen nein.
- **`ensure_directories()` / `assert_writable()`** — legt den Ausgabebaum an und
  beweist die Schreibbarkeit mit einer Schreib-Lese-Loesch-Probe.
- **`model_available()`** — billiger Erreichbarkeitscheck des Repos, bevor der
  mehrere GB grosse Download startet.

In [ ]:
# ── Zelle 1.2 · Zentrale Importe + Umgebungspruefung ─────────────────────
#
# Die einzige Importzelle. Gruppiert nach Modul; wer eine Komponente austauschen
# will, ersetzt genau eine Zeile.

# Konfiguration ──────────────────────────────────────────────────────────
from finetune.config import get_config, MODEL_PROFILES, QUANT_PROFILES

# Umgebung & Diagnose ────────────────────────────────────────────────────
from finetune.utils.environment import (
    report_environment, ensure_directories, assert_writable,
)
from finetune.utils.checks import check_versions, has_valid_adapter, has_valid_merged

# Authentifizierung & Modellbeschaffung ──────────────────────────────────
from finetune.auth.huggingface import authenticate, model_available
from finetune.models.download import ensure_base_model

# Daten ──────────────────────────────────────────────────────────────────
from finetune.data.dataset_loader import prepare_datasets, select_eval_questions
from finetune.evaluation.contamination import detect_contamination

# Preflight & Training ───────────────────────────────────────────────────
from finetune.preflight import preflight
from finetune.training.trainer import train

# Export der Modellvarianten ─────────────────────────────────────────────
from finetune.export.llama_cpp import ensure_llama_cpp
from finetune.export.variants import build_model_variants
from finetune.export.ollama_import import register_ollama_variants

# Evaluierung ────────────────────────────────────────────────────────────
from finetune.evaluation.inference import generate_answers, check_inference_resources
from finetune.evaluation.judge import run_llm_judge
from finetune.evaluation.metrics import score_text_metrics
from finetune.evaluation.statistics import run_statistics, build_long_table
from finetune.evaluation.plots import make_plots
from finetune.evaluation.report import write_report

# Reproduzierbarkeit ─────────────────────────────────────────────────────
from finetune.utils.reproducibility import write_run_manifest

print("✅ Importe vollstaendig\n")

# ── Umgebung pruefen ────────────────────────────────────────────────────
report_environment(CFG)
authenticate(CFG)
ensure_directories(CFG)
assert_writable(CFG)
model_available(CFG)

## Phase 2 · Datenvorbereitung

`prepare_datasets()` spiegelt `train/validation/test_qa.json` aus dem read-only
Input nach `CFG.paths.data_dir`, laedt sie und bringt den Trainingsteil ins
Chat-Format.

**Der System-Prompt kommt aus `CFG.system_prompt` — derselbe, den Phase 6 zur
Evaluierung benutzt.** Das war vorher nicht so: trainiert wurde mit dem
ausfuehrlichen DFSS-Prompt, evaluiert mit *"Answer concisely in 3–4 sentences"*.
Der Unterschied ging ungefiltert in ROUGE-L, BLEU und chrF ein und wurde dort als
Qualitaetsverlust des Modells gelesen.

Zwei Pruefungen laufen mit:

- **Antwortlaengen gegen `num_predict`.** Ist das Generierungslimit kleiner als die
  p90-Referenzantwort, misst die Evaluierung Abschneidung statt Qualitaet. Die
  Zelle nennt dann den passenden Wert.
- **Kontamination.** Exakte Ueberlappung plus Nah-Duplikate ueber Token-Jaccard
  (Schwelle 0.85). Bei 840 Trainings- und 181 Testfragen aus derselben Quelle ist
  reine String-Gleichheit zu schwach — eine umformulierte Frage ist genauso
  kontaminiert wie eine kopierte. Die Pruefung laeuft **hier** und nicht erst in
  der Evaluierung, damit ein kontaminierter Split das Training verhindert, statt
  die Ergebnisse hinterher zu erklaeren.

In [ ]:
# ── Zelle 2.2 · Splits spiegeln, laden, pruefen ──────────────────────────
data = prepare_datasets(CFG)

# data.train / .validation / .test  → Rohdaten (question, answer, type, difficulty)
# data.train_chat / .validation_chat → trainingsfertiges Chat-Format
# data.contamination                → Bericht der Kontaminationspruefung
# data.stats                        → Kennzahlen je Split

## Phase 3 · Preflight

Training ist der teure Teil. Diese Zelle refused ihn, solange nicht bewiesen ist,
dass alles Noetige funktioniert — sie laeuft wenige Minuten und schliesst die
Fehlschlaege aus, die sonst erst nach einer halben GPU-Stunde auftreten.

### Ressourcencheck (`check_resources`)

Misst **tatsaechlich** verfuegbaren RAM (`MemAvailable`), freies VRAM
(`torch.cuda.mem_get_info`) und freien Plattenplatz — und vergleicht das mit dem
geschaetzten Bedarf des in Zelle 0.3 gewaehlten Modells. Gemessen schlaegt dabei
geschaetzt: `CFG.runtime` dient nur als Rueckfallwert und als Dokumentation der
Zielumgebung.

Geprueft werden vier Budgets:

| Budget | Wofuer | Kaggle-Grenze |
|--------|--------|---------------|
| **VRAM** | 4-Bit-Gewichte + Aktivierungen waehrend des Trainings | T4, 16 GB |
| **RAM** | der fp16-Merge in Phase 5 | 32 GB |
| **`/kaggle/working`** | Checkpoint + finale GGUFs (dauerhaft) | ~20 GB |
| **`/tmp` (scratch)** | Merge-Shards, FP16-GGUF, Accelerate-Offload | ~20 GB, **fluechtig** |

`/tmp` ist auf Kaggle *kein* RAM-Laufwerk, sondern liegt auf demselben
Root-Volume — die frueher im Notebook stehende Annahme „Platz satt" traegt bei
groesseren Modellen nicht. Deshalb wird es hier mit eigenem Budget gefuehrt.
Das Offload-Verzeichnis fuer Accelerate (`CFG.paths.offload_dir`) liegt bewusst
dort, damit ausgelagerte Gewichts-Shards nicht `/kaggle/working` fuellen.

Bei Unterdeckung nennt die Meldung die konkrete Stellschraube statt „out of
memory" — etwa die groesste noch passende `max_seq_length`, ein kleineres
`lora.r` oder das naechstkleinere Modellprofil.

### Die sechs Integritaetschecks

1. **Versionen** — gegen `CFG.model`, nicht mehr gegen ein fest verdrahtetes
   `Gemma4Config`. Ein Profil ohne Sonderanforderung laeuft ohne Versionshuerde.
2. **Checkpoint** — `ensure_base_model()` ist offline-first: vorhandene, zum
   Profil passende Kopie wird wiederverwendet, sonst einmaliger Download
   (ohne `*.gguf`/`original/*`, das spart bei manchen Repos mehrere GB).
3. **Chat-Template** — muss `system/user/assistant` akzeptieren. Zusaetzlich
   wird geprueft, ob der System-Prompt im gerenderten Text ueberhaupt auftaucht.
4. **4-Bit-Ladevorgang** — mit exakt der Konfiguration, die das Training nutzt.
5. **LoRA-Zielmodule** — gegen das geladene Modell; bei multimodalen Modellen
   werden die Vision-/Audio-Tower ausgeschlossen.
6. **Tokenisierung + ein Generierungsschritt** — danach wird die GPU vollstaendig
   freigegeben, damit das Training sauber startet.

In [ ]:
# ── Zelle 3.2 · Ressourcenbudget und Integritaetschecks ──────────────────
preflight_report = preflight(CFG)

# strict=False meldet Probleme, bricht aber nicht ab (nur zum Erkunden):
# preflight_report = preflight(CFG, strict=False)

## Phase 4 · LoRA-Fine-Tuning

`train(CFG, data)` bezieht **jeden** Parameter aus der Konfiguration — Epochen,
Batchgroessen, Lernrate, Warmup, LoRA-Raenge, Sequenzlaenge, Seed, Speicher- und
Auswertungsstrategie. In der Altfassung rief `train_model()` intern
`run_training(epochs=3, batch_size=1, grad_accum=8, lr=2e-4)` mit Literalen auf;
die Werte aus `settings.py` waren damit wirkungslos.

**Was hier passiert:**

- **QLoRA:** Die Basisgewichte liegen eingefroren in 4 Bit (NF4, Double Quant),
  trainiert werden nur die Low-Rank-Adapter auf den Projektionen. Das macht das
  Fine-Tuning eines Mehr-Milliarden-Parameter-Modells auf einer T4 ueberhaupt
  moeglich.
- **Gradient Checkpointing** tauscht etwa 20 % Rechenzeit gegen einen grossen
  Teil des Aktivierungsspeichers. `use_cache` wird dafuer abgeschaltet.
- **Validierung und Early Stopping:** Der Validierungssplit wird nach jeder
  Epoche ausgewertet; bei `early_stopping` bricht der Lauf ab, wenn sich
  `eval_loss` ueber `early_stopping_patience` Auswertungen nicht verbessert, und
  laedt am Ende den besten Checkpoint. In der Altfassung lag der
  Validierungssplit ungenutzt in einem Kommentar.
- **Reproduzierbarkeit:** `seed` und `data_seed` aus `CFG.training`.

**Wiederholtes Ausfuehren ist sicher.** Die Zelle entscheidet selbst:

| Situation | Verhalten |
|-----------|-----------|
| `smoke_test=True` | Kurzlauf ueber `smoke_max_steps` Schritte |
| `force_retrain=True` | voller Lauf, vorhandener Adapter wird ersetzt |
| Adapter da + `resume_training=True` | Fortsetzung am letzten Checkpoint |
| Adapter da | uebersprungen, vorhandener Adapter wird wiederverwendet |
| sonst | voller Lauf |

Geschrieben werden der Adapter nach `CFG.paths.adapter_dir` und daneben
`training_meta.json` mit Schrittzahl, Loss-Verlauf, bestem Checkpoint,
Token-Statistik und der effektiven Batchgroesse — die Grundlage fuer das
Run-Manifest in Phase 7.

Ein Hinweis, den die Zelle selbst gibt: Wenn Trainingsbeispiele an
`max_seq_length` abgeschnitten werden, sieht das Modell deren Antwortende nie.
Bei diesem Datensatz (Antwort-Median ~520 Zeichen) reichen 512 Tokens; bei
laengeren Antworten meldet die Zelle den Anteil abgeschnittener Beispiele.

In [ ]:
# ── Zelle 4.2 · Fine-Tuning ──────────────────────────────────────────────
training_meta = train(CFG, data)

# training_meta["status"]        "trained" | "skipped"
# training_meta["global_step"]   tatsaechlich gelaufene Schritte
# training_meta["best_metric"]   bester eval_loss (bei Early Stopping)

## Phase 5 · Die drei Modellvarianten

Hier entsteht der Kern des Experiments. Drei Varianten, die sich jeweils in
**genau einer** Eigenschaft unterscheiden — nur so lassen sich Quantisierungs-
fehler und Fine-Tuning-Effekt getrennt beziffern:

| Variante | Herkunft | Der Vergleich zur Vorgaengerin zeigt |
|----------|----------|--------------------------------------|
| `original` | Basismodell → FP16-GGUF | — (Referenz) |
| `base_quant` | **dieselbe FP16-Datei**, quantisiert | reinen **Quantisierungsfehler** |
| `finetuned_quant` | LoRA-Merge → FP16 → quantisiert | **Fine-Tuning-Effekt** unter Quantisierung |

Zwei Details, die den Vergleich erst sauber machen:

- **`base_quant` wird aus `original` quantisiert**, nicht ein zweites Mal aus
  dem HF-Checkpoint konvertiert. Damit stammen beide nachweislich aus derselben
  Datei und unterscheiden sich ausschliesslich durch die Quantisierung.
- **Der Merge laeuft auf fp16, nicht auf einem 4-bit geladenen Basismodell.**
  Die Altfassung fusionierte in NF4-dequantisierte Gewichte und buk damit den
  Quantisierungsfehler des Trainings in die Gewichte ein — nicht mehr trennbar
  von dem, was hier gemessen werden soll.

### Reihenfolge und Plattenplatz

Die Stufenfolge ist vom Kaggle-Budget diktiert, nicht von der Logik. Ein
FP16-GGUF eines 4B-Modells ist ~8 GB gross und entsteht, waehrend der
HF-Checkpoint (~8 GB) noch liegt — zusammen mit den Artefakten passt das nicht
in 20 GB. Deshalb:

1. `original` entsteht **in `/tmp`**, nicht direkt am Ziel.
2. `base_quant` wird daraus quantisiert und geht sofort nach `artifacts_dir`.
3. Der Merge (letzter Schritt, der den Checkpoint braucht) laeuft nach `/tmp`.
4. **Jetzt** faellt der Checkpoint weg, und erst dann wandert das FP16-GGUF an
   sein Ziel. Genau hier laege sonst das Maximum.
5. `finetuned_quant`: Merge → FP16 → quantisieren, Zwischenprodukte sofort weg.

`CFG.export_stage_peaks()` rechnet das vor; Zelle 5.2 zeigt die Zahlen. Fuer
Gemma 4B ergibt sich ein Spitzenbedarf von rund 13 GB in `/kaggle/working` und
16 GB in `/tmp` — beides mit Reserve. Ein 8B-Modell passt mit drei Varianten
**nicht** auf Kaggle; `CFG.validate()` meldet das bereits in Zelle 0.3.

Der Ollama-Blobstore liegt bewusst in `/tmp`: `ollama create` kopiert jedes
GGUF ein zweites Mal, und diese Kopie ist aus `artifacts_dir` in Sekunden
wiederherstellbar. Sie darf die 20 GB in `/kaggle/working` nicht belegen.

### Wiederaufsetzen

Jede fertige, gueltige GGUF-Datei wird uebersprungen. Ein Session-Neustart
mitten in Phase 5 kostet damit nur die fehlenden Stufen.
`CFG.export.force_rebuild = True` erzwingt einen kompletten Neubau.

In [ ]:
# ── Zelle 5.2 · llama.cpp bereitstellen (idempotent) ─────────────────────
# Wiederverwenden, sonst aus dem Input-Dataset kopieren, sonst klonen + bauen.
ensure_llama_cpp(CFG)

# Plattenbudget der Exportstufen — dieselbe Rechnung, die CFG.validate() nutzt
for stage, gb in CFG.export_stage_peaks().items():
    if stage.startswith("stage"):
        print(f"  {stage:<32} {gb:6.1f} GB")

In [ ]:
# ── Zelle 5.3 · Die drei Varianten erzeugen ──────────────────────────────
variants = build_model_variants(CFG)

# variants["original"].gguf_path      FP16-GGUF (Referenz)
# variants["base_quant"].size_gb      Groesse der quantisierten Basis
# variants["finetuned_quant"].built   True = in diesem Lauf gebaut

In [ ]:
# ── Zelle 5.4 · In Ollama registrieren ───────────────────────────────────
# Alle drei Varianten laufen danach ueber dieselbe Schnittstelle — die
# Voraussetzung dafuer, dass Antwortzeit und Tokenzahlen in Phase 6
# ueberhaupt vergleichbar sind.
variants = register_ollama_variants(CFG, variants)

## Phase 6 · Evaluierung

Vier Schritte, jeder mit eigener Ergebnisdatei in `results/`, jeder einzeln
wiederholbar und fortsetzbar. Das ist auf Kaggle wichtig: Eine Session endet
nach einigen Stunden, und die Antwortgenerierung ist der laengste Abschnitt der
ganzen Pipeline.

### Was hier verglichen wird

Die drei Varianten unterscheiden sich in **genau einer** Eigenschaft. Alles
andere ist konstant gehalten — derselbe System-Prompt aus
`CFG.training.system_prompt`, dieselben Fragen in derselben Reihenfolge,
dieselben Sampling-Parameter, dieselbe Schnittstelle. Nur so misst der
Vergleich das, was er messen soll.

Im Altstand war genau das nicht der Fall: trainiert wurde mit dem
ausfuehrlichen DFSS-Prompt, evaluiert mit *"Answer concisely in 3–4
sentences"* und an einer dritten Stelle mit einer deutschen Fassung, bei
`num_predict = 200`. Der Prompt-Unterschied und die Abschneidung gingen
ungefiltert in ROUGE-L und BLEU ein und wurden dort als Qualitaetsverlust des
Modells gelesen.

### Die vier Schritte

| Zelle | Was | Ergebnis |
|---|---|---|
| 6.2 | Antworten aller Varianten | `answers_<variante>.json` |
| 6.3 | LLM-Judge, 0–4 je Dimension | `judge_<variante>.json` |
| 6.4 | BERTScore, ROUGE-L, BLEU, chrF | `metrics_<variante>.json` |
| 6.5 | Bootstrap-KIs, Wilcoxon, Diagramme | `evaluation_report.json`, CSVs, PNGs |

**Antwortzeit und Tokenzahl** werden bei jeder Antwort aus der Ollama-Antwort
mitgeschrieben (`eval_count`, `prompt_eval_count`, `eval_duration`) und pro
Variante getrennt ausgewiesen. Im Altstand waren diese Felder vorhanden und
wurden verworfen.

**Keine stillen Ausfaelle.** Leere Antworten, Fehler und fehlende Tokenzahlen
bekommen einen eigenen Status und bleiben in den Dateien. Sie werden gezaehlt
und im Bericht genannt, statt einfach aus dem Mittelwert zu verschwinden.

In [ ]:
# ── Zelle 6.2 · Antworten aller Varianten erzeugen ───────────────────────
# Ruft intern check_resources() und prueft zusaetzlich, ob die groesste
# Variante geladen werden kann. Fortsetzbar: bereits beantwortete Fragen
# werden uebersprungen.
answers = generate_answers(CFG)

# force=True erzwingt eine komplette Neugenerierung:
# answers = generate_answers(CFG, force=True)

In [ ]:
# ── Zelle 6.3 · LLM-Judge (0–4 je Dimension) ─────────────────────────────
# Absolute Bewertung einer einzelnen Antwort gegen die Referenz — kein
# Paarvergleich. Der Judge erfaehrt nicht, von welcher Variante die Antwort
# stammt. Antworten ohne verwertbaren Inhalt bekommen automatisch 0.
judgements = run_llm_judge(CFG, answers)

In [ ]:
# ── Zelle 6.4 · Textmetriken je Beispiel ─────────────────────────────────
# Einzelwerte, nicht nur Korpuswerte: Bootstrap und der gepaarte Wilcoxon-Test
# brauchen sie. BERTScore laeuft in Baetchen (CFG.evaluation.bertscore_batch_size),
# damit das Backbone den Arbeitsspeicher nicht sprengt.
metrics = score_text_metrics(CFG, answers)

In [ ]:
# ── Zelle 6.5 · Statistik, Diagramme, Bericht ────────────────────────────
report = run_statistics(CFG, answers, metrics, judgements)
plots = make_plots(report, CFG)
write_report(report, CFG, plots, data=data)

# report["summary"][variante][metrik]  → mean, ci_low, ci_high, std, median, n
# report["comparisons"]                → Wilcoxon-Ergebnisse mit Effektstaerke

## Phase 7 · Reproduzierbarkeit

`write_run_manifest()` haelt fest, was diesen Lauf ausgemacht hat: die
vollstaendige Konfiguration als JSON, die installierten Paketversionen, die
Trainingskennzahlen und die erzeugten Artefakte mit Groesse.

`with_hashes=True` berechnet zusaetzlich SHA-256 ueber die GGUF-Dateien. Das
dauert bei 13 GB einige Minuten und lohnt sich nur, wenn die Artefakte
weitergegeben werden — dann laesst sich spaeter beweisen, dass eine Datei die
ist, mit der die Zahlen im Bericht entstanden sind.

In [ ]:
# ── Zelle 7.2 · Manifest, Konfiguration und Requirements schreiben ───────
manifest = write_run_manifest(CFG)

# Mit Pruefsummen (dauert bei 13 GB einige Minuten):
# manifest = write_run_manifest(CFG, with_hashes=True)

---

## Anhang A · Diagnose

Standardmaessig ausgeschaltet. Aktivieren ueber `CFG.debug = True` in Zelle 0.3
oder direkt in der Zelle unten.

Ersetzt die verstreuten Diagnosezellen des Altstands (dort Zellen 12, 25, 37,
58, 66, 73, 78 — allein Zelle 37 hatte 176 Zeilen reine Ausgabe).

In [ ]:
# ── Zelle A.2 · Diagnose (nur bei CFG.debug = True) ──────────────────────
if CFG.debug:
    import shutil, subprocess
    from pathlib import Path

    print("── Speicherplatz " + "─" * 50)
    for label, path in (("working", CFG.paths.base_dir),
                        ("scratch", CFG.paths.scratch_dir),
                        ("input  ", CFG.paths.input_dir)):
        probe = path
        while not probe.exists() and probe != probe.parent:
            probe = probe.parent
        usage = shutil.disk_usage(probe)
        print(f"  {label}: {usage.free / 1e9:6.1f} GB frei von "
              f"{usage.total / 1e9:6.1f} GB   ({path})")

    print("\n── Belegung unter base_dir " + "─" * 40)
    if CFG.paths.base_dir.exists():
        for item in sorted(CFG.paths.base_dir.iterdir()):
            size = (sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
                    if item.is_dir() else item.stat().st_size)
            print(f"  {size / 1e9:7.2f} GB  {item.name}")

    print("\n── RAM " + "─" * 60)
    print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)

    print("── Ollama " + "─" * 57)
    print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

    print("── Aktive Konfiguration " + "─" * 43)
    print(CFG.summary())
else:
    print("Anhang A ist deaktiviert (CFG.debug = False).")

---

## Anhang B · Artefakte als Kaggle-Dataset sichern

Standardmaessig ausgeschaltet. Aktivieren ueber `CFG.publish_artifacts = True`.

Sinnvoll am Ende einer Session: `/kaggle/working` ueberlebt das Ende des
Kernels nicht zuverlaessig, und die GGUF-Dateien neu zu bauen kostet eine
Stunde. Als Dataset gesichert lassen sie sich in der naechsten Session ueber
`CFG.export.llama_cpp_input_copy` beziehungsweise als Input-Dataset direkt
wieder einbinden.

Ersetzt die neun verstreuten Upload- und ZIP-Zellen des Altstands (Zellen 4, 6,
9, 10, 26, 27, 28, 52, 69), darunter zwei vollstaendige Kopien derselben
200-MB-Chunk-Logik.

In [ ]:
# ── Zelle B.2 · Artefakte veroeffentlichen (nur bei CFG.publish_artifacts) ──
if CFG.publish_artifacts:
    import shutil
    from pathlib import Path
    import kagglehub

    KAGGLE_USER = "feyssal"          # anpassen
    handle = f"{KAGGLE_USER}/{CFG.model.slug}-{CFG.dataset_tag}-artifacts"

    staging = CFG.paths.scratch_dir / "kaggle_upload"
    if staging.exists():
        shutil.rmtree(staging)       # nur dieser eine, klar benannte Pfad
    staging.mkdir(parents=True)

    for variant in CFG.active_variants:
        source = CFG.gguf_paths[variant]
        if source.exists():
            print(f"  kopiere {source.name} ({source.stat().st_size / 1e9:.1f} GB) …")
            shutil.copy2(source, staging / source.name)
    for name in ("report.md", "evaluation_report.json", "per_sample.csv",
                 "summary.csv", "wilcoxon.csv", "run_manifest.json"):
        source = CFG.paths.results_dir / name
        if source.exists():
            shutil.copy2(source, staging / name)

    print(f"\nVeroeffentliche als {handle} …")
    kagglehub.dataset_upload(
        handle=handle,
        local_dataset_dir=str(staging),
        version_notes=f"{CFG.model.hf_repo_id} · {CFG.quantization.key} · "
                      f"{len(CFG.active_variants)} Varianten",
    )
    shutil.rmtree(staging, ignore_errors=True)
    print("✅ Hochgeladen.")
else:
    print("Anhang B ist deaktiviert (CFG.publish_artifacts = False).")